# 03. Tier 3 Lightweight Ontology & Stratified LLM Sampling

### Architectural Purpose
While our **Tier 2 Semantic Layer (`silver_sessions`)** compresses 42.45M raw rows into clean session episodes, flat SQL tables still require complex self-joins and implicit business-rule prompts to answer **multi-hop causal questions** (e.g., *"Which brands lose price-sensitive shoppers to competitors after cart abandonment?"*).

In this notebook, we build our **Tier 3 Lightweight Ontology** in PySpark + Python (without external graph database overhead):
1. **Materialize Domain Objects (`workspace.default.ont_nodes`):** Typed entities for `BrandEntity`, `CategoryEntity`, and `SessionEpisode`.
2. **Materialize Causal & Behavioral Edges (`workspace.default.ont_edges`):** Explicit typed relationships (`SWITCHED_BRAND_TO`, `ABANDONED_BRAND_CART`, `PURCHASED_BRAND`, `BYPASSED_CART_DIRECT_BUY`) aggregated across non-bot sessions.
3. **Verify Pydantic Schema Compliance:** Validate sample nodes and edges against `src/schemas.py` to guarantee type safety for Agent C's graph traversal tools.
4. **Build Stratified Evaluation Sample (`workspace.default.gold_sampled_sessions`):** Draw a deterministic, balanced sample of **3,000 non-bot sessions** across all 5 funnel outcomes (`standard_purchase`, `direct_purchase_no_cart`, `cart_abandoned`, `multi_view_browse`, `single_event_bounce`) for structured LLM intent tagging and benchmark evaluation.


In [0]:
import sys
# Add the repo's src/ directory to Python path so Databricks imports our Pydantic schemas cleanly
sys.path.append("../src")

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from schemas import OntologyNode, OntologyEdge, SessionIntent

print("Successfully imported Pydantic v2 schemas from ../src/schemas.py!")


In [0]:
# Load non-bot stateful sessions from Tier 2
silver_clean = (
    spark.table("workspace.default.silver_sessions")
    .filter(F.col("is_bot_suspect") == False)
)

# 1. Edge Type: SWITCHED_BRAND_TO (Competitive brand leakage within a session)
# Captures sessions where a user started viewing primary_brand and ended on a different final_brand
brand_switch_edges = (
    silver_clean
    .filter(
        (F.col("brand_switch_count") > 0) &
        (F.col("primary_brand") != "unknown") &
        (F.col("final_brand") != "unknown") &
        (F.col("primary_brand") != F.col("final_brand"))
    )
    .groupBy(
        F.concat(F.lit("brand:"), F.col("primary_brand")).alias("source_id"),
        F.concat(F.lit("brand:"), F.col("final_brand")).alias("target_id"),
        F.col("primary_category").alias("context_category")
    )
    .agg(
        F.lit("SWITCHED_BRAND_TO").alias("relation_type"),
        F.count("*").alias("weight"),
        F.round(F.sum("session_revenue"), 2).alias("revenue_impact")
    )
)

# 2. Edge Type: ABANDONED_BRAND_CART (Category -> Brand friction link)
cart_abandon_edges = (
    silver_clean
    .filter((F.col("funnel_outcome") == "cart_abandoned") & (F.col("primary_brand") != "unknown"))
    .groupBy(
        F.concat(F.lit("category:"), F.col("primary_category")).alias("source_id"),
        F.concat(F.lit("brand:"), F.col("primary_brand")).alias("target_id"),
        F.col("primary_category").alias("context_category")
    )
    .agg(
        F.lit("ABANDONED_BRAND_CART").alias("relation_type"),
        F.count("*").alias("weight"),
        F.round(F.sum(F.col("avg_price_viewed") * F.col("cart_count")), 2).alias("revenue_impact")
    )
)

# 3. Edge Type: BYPASSED_CART_DIRECT_BUY (Encodes domain axiom: 1-click purchases skip cart events)
direct_buy_edges = (
    silver_clean
    .filter((F.col("funnel_outcome") == "direct_purchase_no_cart") & (F.col("final_brand") != "unknown"))
    .groupBy(
        F.concat(F.lit("category:"), F.col("primary_category")).alias("source_id"),
        F.concat(F.lit("brand:"), F.col("final_brand")).alias("target_id"),
        F.col("primary_category").alias("context_category")
    )
    .agg(
        F.lit("BYPASSED_CART_DIRECT_BUY").alias("relation_type"),
        F.count("*").alias("weight"),
        F.round(F.sum("session_revenue"), 2).alias("revenue_impact")
    )
)

# 4. Edge Type: PURCHASED_BRAND (Standard completed conversion link)
purchase_edges = (
    silver_clean
    .filter((F.col("funnel_outcome") == "standard_purchase") & (F.col("final_brand") != "unknown"))
    .groupBy(
        F.concat(F.lit("category:"), F.col("primary_category")).alias("source_id"),
        F.concat(F.lit("brand:"), F.col("final_brand")).alias("target_id"),
        F.col("primary_category").alias("context_category")
    )
    .agg(
        F.lit("PURCHASED_BRAND").alias("relation_type"),
        F.count("*").alias("weight"),
        F.round(F.sum("session_revenue"), 2).alias("revenue_impact")
    )
)

# Union all typed relationship edges and persist to Delta
ont_edges = (
    brand_switch_edges
    .unionByName(cart_abandon_edges)
    .unionByName(direct_buy_edges)
    .unionByName(purchase_edges)
)

(
    ont_edges.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.ont_edges")
)

# Validate top edges with Pydantic OntologyEdge model
sample_edges = spark.table("workspace.default.ont_edges").orderBy(F.desc("weight")).limit(5).collect()
validated_edges = [OntologyEdge(**row.asDict()) for row in sample_edges]
print(f"Validated {len(validated_edges)} sample edges via Pydantic. Top edge: {validated_edges[0]}")

spark.table("workspace.default.ont_edges").groupBy("relation_type").agg(
    F.count("*").alias("distinct_edges"),
    F.sum("weight").alias("total_edge_weight"),
    F.round(F.sum("revenue_impact"), 2).alias("total_revenue_impact")
).show(truncate=False)


In [0]:
# Draw a deterministic, balanced sample of 600 sessions per funnel_outcome (5 outcomes * 600 = 3,000 sessions).
# Using a fixed seed (42) ensures 100% reproducibility for conference & paper reviewers.
w_strat = Window.partitionBy("funnel_outcome").orderBy(F.rand(seed=42))

gold_sampled_sessions = (
    silver_clean
    .withColumn("rn", F.row_number().over(w_strat))
    .filter(F.col("rn") <= 600)
    .drop("rn")
)

(
    gold_sampled_sessions.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_sampled_sessions")
)

print(f"Materialized gold_sampled_sessions with {spark.table('workspace.default.gold_sampled_sessions').count():,} balanced sessions.")

spark.table("workspace.default.gold_sampled_sessions").groupBy("funnel_outcome").agg(
    F.count("*").alias("sampled_sessions"),
    F.round(F.avg("total_events"), 1).alias("avg_events"),
    F.round(F.avg("brand_switch_count"), 2).alias("avg_brand_switches"),
    F.round(F.avg("price_step_down_count"), 2).alias("avg_price_step_downs")
).show(truncate=False)
